# AquaPINN Colab — FishHeart Full Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/03_FishHeart_Full_Pipeline.ipynb)


Run one FishHeart tag through raw receiver decoding, clock synchronization, and trajectory tracking. Configure the main deployment and solver settings below. Results are saved in a desktop-style Full Study folder with run metadata and per-tag track output.

This workflow may take substantial time. Use only receiver data and GPS/reference files approved for processing in hosted Colab.

## 1. Source, runtime, and persistent output

Set REPO_URL after the public repository has been created. Input fields default to the bundled FishHeart example and can be changed to files in Drive.

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, uuid

REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_REF = "main"

def find_repo():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "AquaPINN_Tracker3D").is_dir() and (p / "Examples").is_dir():
            return p
    return None

REPO_DIR = find_repo()
if REPO_DIR is None:
    if not REPO_URL:
        raise RuntimeError("Set REPO_URL to the public repository URL, then rerun this cell.")
    REPO_DIR = Path("/content/aquapinn-source")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                        REPO_URL, str(REPO_DIR)], check=True)
    if not (REPO_DIR / "AquaPINN_Tracker3D").is_dir():
        raise RuntimeError("Repository is missing AquaPINN_Tracker3D/")

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
    OUTPUT_PARENT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    IN_COLAB = False
    OUTPUT_PARENT = Path.cwd() / "AquaPINN_Colab_Runs"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)

def new_run(label):
    run_id = uuid.uuid4().hex
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    path = OUTPUT_PARENT / f"{label}_{stamp}_{run_id[:8]}"
    path.mkdir(parents=True, exist_ok=False)
    return path, run_id

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, default=str) + "\n", encoding="utf-8")

sys.path.insert(0, str(REPO_DIR))

print("Project:", REPO_DIR)
print("Persistent output parent:", OUTPUT_PARENT)


## 2. Install dependencies

In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six tomli-w

In [ ]:
import torch, tomllib
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.11 or later is recommended.")

In [ ]:
import pandas as pd

def track_digest(path):
    path = Path(path)
    frame = pd.read_csv(path)
    return {
        "csv_sha256": sha256(path),
        "rows": int(len(frame)),
        "columns": [str(c) for c in frame.columns],
        "first_datetime": str(frame["datetime"].dropna().iloc[0]) if "datetime" in frame and frame["datetime"].notna().any() else None,
        "last_datetime": str(frame["datetime"].dropna().iloc[-1]) if "datetime" in frame and frame["datetime"].notna().any() else None,
    }

def make_study(name, method, samples, parameters, tag_file, selected_rows,
               coordinate_frame, source_config):
    table = pd.read_csv(tag_file)
    selected = {int(i) for i in selected_rows}
    tags = []
    for row, record in table.iterrows():
        code = str(record.get("Code", f"Tag{row}")).strip()
        try:
            pri = float(record.get("PRI", 0))
        except (TypeError, ValueError):
            pri = 0.0
        tags.append({"code": code, "row": int(row), "nominal_pri": pri,
                     "role": "fish" if int(row) in selected else "unassigned",
                     "selected": int(row) in selected, "gps": None})
    return {
        "format": "aquapinn-study", "version": 1, "name": name,
        "workflow": "fish", "method": method,
        "nnplus_samples": max(2, int(samples)),
        "parameters": {k: (v if isinstance(v, list) else [v]) for k, v in parameters.items()},
        "tags": tags, "unknown_parameters": {}, "source_config": str(source_config),
        "import_notes": ["Created by an AquaPINN Colab workflow."],
        "coordinate_frame": coordinate_frame, "sync_support_reviewed": True,
        "gps_draft": {"file": "", "utc_offset": 0, "matching_frame": False,
                       "confirmed": False, "start": "", "end": "", "target_codes": []},
        "evaluation": {"window_seconds": 90, "step_seconds": 1,
                       "max_gap_seconds": 5, "constant_pri_confirmed": False},
        "example": False,
    }


## 3. Study setup and main parameters

Review the metadata paths, recording folder, selected tag, time window, timezone, temperature, dimension, solver, and output parent. The workflow runs decode and synchronization because its inputs are raw receiver recordings.

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

FISH_RUN = REPO_DIR / "Examples/fishHeart/run"
PARAMS_FILE = FISH_RUN / "params.toml"
with PARAMS_FILE.open("rb") as stream:
    PARAMS = tomllib.load(stream)

def bundled_path(key):
    value = PARAMS[key][0]
    return str((FISH_RUN / value).resolve()) if value else ""

SETUP = {
    "phone_file": bundled_path("phoneFile"),
    "tag_file": bundled_path("tagFile"),
    "recordings_folder": bundled_path("dataFolder"),
    "test_file": bundled_path("testFile"),
    "gps_file": bundled_path("GPSFile"),
    "time_start": PARAMS["time_start"][0],
    "time_end": PARAMS["time_end"][0],
    "time_zone_hours": int(PARAMS["timeZone"][0]),
    "temperature_c": float(PARAMS["temperature_ref"][0]),
    "horizontal_only": PARAMS["ignoreZ"][0].lower() == "true",
    "tag_id": int(PARAMS["TagID"][0]),
    "method": "NN",
    "nnplus_samples": 5,
    "device": "Automatic",
}
display(pd.DataFrame(SETUP.items(), columns=["Parameter", "Value"]))
TAG_TABLE = pd.read_csv(SETUP["tag_file"])
print("Catalogue codes:", ", ".join(TAG_TABLE["Code"].astype(str)))

Edit the SETUP dictionary above for your study. For custom deployments, update all receiver/tag/GPS metadata paths together. Use paths under mounted Drive for custom files. TagID is the zero-based row in the selected tag catalogue.

In [ ]:
from pathlib import Path
import tomli_w

PHONE_FILE = Path(SETUP["phone_file"]).expanduser().resolve()
TAG_FILE = Path(SETUP["tag_file"]).expanduser().resolve()
DATA_DIR = Path(SETUP["recordings_folder"]).expanduser().resolve()
TEST_FILE = Path(SETUP["test_file"]).expanduser().resolve() if SETUP["test_file"] else None
GPS_FILE = Path(SETUP["gps_file"]).expanduser().resolve() if SETUP["gps_file"] else None
TAG_ID = int(SETUP["tag_id"])
TAG_TABLE = pd.read_csv(TAG_FILE)
if not PHONE_FILE.is_file() or not TAG_FILE.is_file() or not DATA_DIR.is_dir():
    raise FileNotFoundError("Required receiver geometry, tag catalogue, or recording folder is missing.")
receiver_files = sorted(DATA_DIR.glob("SR*.csv"))
if not receiver_files:
    raise FileNotFoundError(f"No SR*.csv recordings found in {DATA_DIR}")
if TEST_FILE and not TEST_FILE.is_file():
    raise FileNotFoundError(TEST_FILE)
if GPS_FILE and not GPS_FILE.is_file():
    raise FileNotFoundError(GPS_FILE)
if not 0 <= TAG_ID < len(TAG_TABLE):
    raise ValueError("TagID is outside the selected tag catalogue.")

METHOD = SETUP["method"]
ENGINE_METHOD = "AML" if METHOD == "AML" else "NN"
SAMPLES = 1 if METHOD != "NNplus" else int(SETUP["nnplus_samples"])
TAG_CODE = str(TAG_TABLE.iloc[TAG_ID]["Code"]).strip()
RUN_DIR, RUN_ID = new_run("FishHeart")
PROCESSING = RUN_DIR / "processing"
PARAMS.update({
    "phoneFile": [str(PHONE_FILE)], "tagFile": [str(TAG_FILE)],
    "dataFolder": [str(DATA_DIR)], "testFile": [str(TEST_FILE) if TEST_FILE else ""],
    "GPSFile": [str(GPS_FILE) if GPS_FILE else ""],
    "output_folder": [str(PROCESSING)], "TagID": [TAG_ID],
    "time_start": [SETUP["time_start"]], "time_end": [SETUP["time_end"]],
    "timeZone": [int(SETUP["time_zone_hours"])],
    "temperature_ref": [float(SETUP["temperature_c"])],
    "ignoreZ": ["True" if SETUP["horizontal_only"] else "False"],
    "trackMethod": [ENGINE_METHOD], "nSamples": [SAMPLES],
})
EFFECTIVE_PARAMS = RUN_DIR / "params_effective.toml"
EFFECTIVE_PARAMS.write_text(tomli_w.dumps(PARAMS), encoding="utf-8")
write_json(RUN_DIR / "params_effective.json", PARAMS)
TRACK_REL = f"processing/Track/track/track_Tag{TAG_ID}_{TAG_CODE}_{METHOD}.csv"
print("Selected:", TAG_CODE, "TagID", TAG_ID, "|", METHOD, "| fits:", SAMPLES)
print("Recordings:", len(receiver_files), "| time:", SETUP["time_start"], "to", SETUP["time_end"])
print("Run folder:", RUN_DIR)

## 4. Decode, synchronize, and track

In [ ]:
import subprocess
from copy import deepcopy

study = make_study(
    "FishHeart Colab", METHOD, SAMPLES, PARAMS, TAG_FILE, [TAG_ID],
    "FishHeart receiver coordinates as supplied; confirm projection and datum for scientific interpretation.",
    RUN_DIR / "params_effective.json",
)
study["source_config"] = str(RUN_DIR / "params_effective.json")
tag_entry = deepcopy(study["tags"][TAG_ID])
record = {
    "format": "aquapinn-study-run", "version": 1, "run_id": RUN_ID,
    "state": "starting", "method": METHOD, "study": study,
    "jobs": [{**tag_entry, "method": METHOD, "state": "queued", "track_csv": TRACK_REL}],
}
write_json(RUN_DIR / "study.json", study)
write_json(RUN_DIR / "run.json", record)
input_files = [PHONE_FILE, TAG_FILE, *receiver_files]
input_files += [p for p in [TEST_FILE, GPS_FILE] if p]
write_json(RUN_DIR / "source-input-hashes.json", {str(p): sha256(p) for p in input_files})
write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "running",
                                      "stage": "Decode, synchronize, and track"})
env = os.environ.copy()
if SETUP["device"] == "CPU only":
    env["CUDA_VISIBLE_DEVICES"] = ""
launcher_source = (FISH_RUN / "main_FishHeart.py").read_text(encoding="utf-8")
launcher_source = launcher_source.replace(
    'SCRIPT_DIR   = Path(__file__).resolve().parent',
    f'SCRIPT_DIR   = Path(r"{FISH_RUN}")')
launcher_source = launcher_source.replace(
    'param_file = str(SCRIPT_DIR / "params.toml")',
    f'param_file = r"{RUN_DIR / "params_effective.toml"}"')
launcher = RUN_DIR / "colab_launcher.py"
launcher.write_text(launcher_source, encoding="utf-8")
with (RUN_DIR / "console.log").open("w", encoding="utf-8") as log:
    completed = subprocess.run([sys.executable, str(launcher)], cwd=FISH_RUN,
                               env=env, stdout=log, stderr=subprocess.STDOUT, text=True)

track_csv = RUN_DIR / TRACK_REL
if completed.returncode != 0 or not track_csv.is_file() or not Path(str(track_csv) + ".pickle").is_file():
    record.update(state="failed")
    record["jobs"][0].update(state="failed", error=f"Pipeline exit code {completed.returncode}")
    write_json(RUN_DIR / "run.json", record)
    write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "failed",
                                          "exit_code": completed.returncode})
    raise RuntimeError(f"FishHeart pipeline failed; review {RUN_DIR / 'console.log'}")

summary = track_digest(track_csv)
record["jobs"][0].update(state="completed", track_sha256=summary["csv_sha256"], summary=summary)
record.update(state="completed", device=("CPU" if SETUP["device"] == "CPU only" else
                                         ("CUDA" if torch.cuda.is_available() else "CPU")))
write_json(RUN_DIR / "run.json", record)
write_json(RUN_DIR / "worker.json", {"run_id": RUN_ID, "state": "completed",
                                      "stage": "Completed", "track_csv": TRACK_REL,
                                      "track_sha256": summary["csv_sha256"]})
print("Completed:", track_csv)
print("Desktop Full Study run record:", RUN_DIR / "run.json")

## 5. Review and retain results

The run folder contains a Full Study run record, selected parameters, source/input hashes, console and engine logs, decoder/synchronization intermediates, the track CSV and pickle sidecar, and figures.

In [ ]:
from IPython.display import display, Image
track = pd.read_csv(track_csv)
display(track.head())
plots = sorted(PROCESSING.rglob("*.jpg"))
print("Figures:", len(plots))
for path in plots[:4]:
    display(Image(filename=str(path)))
print("Complete run folder:", RUN_DIR)